# A seeded scientific workflow

This offline notebook uses a generated dataset and bounded settings. It demonstrates data profiling, a small AutoML search, repeated nested-CV evaluation, and model-specific feature interpretation. Inputs and sampling are seeded, but search/model results can still vary across runs or environments. No download, API service, or external dataset is required.

Run from the repository root with `jupyter lab examples/notebooks/scientific_workflow.ipynb`. Results are examples, not scientific conclusions or model-quality guarantees.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import get_scorer
from sklearn.model_selection import train_test_split

from SKSurrogate import AML, dataset_profile, mltrack

SEED = 7
X, y = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    random_state=SEED,
)
features = [f"feature_{index}" for index in range(X.shape[1])]
frame = pd.DataFrame(X, columns=features).assign(target=y)
train, validation = train_test_split(
    frame, test_size=0.25, random_state=SEED, stratify=frame["target"]
)
train = train.reset_index(drop=True)
validation = validation.reset_index(drop=True)
print(f"Training rows: {len(train)}; validation rows: {len(validation)}")

## Inspect data before modeling

The profile reports distributions, missingness, and exploratory feature/target associations. Associations are diagnostic signals, not proof of causation, leakage, or redundant scientific mechanisms.

In [ ]:
profile = dataset_profile(
    train,
    target="target",
    max_rows=500,
    max_features=10,
    max_categories=10,
    random_state=SEED,
)
display(pd.DataFrame(profile["columns"])[["column", "kind", "missing_rate", "unique_count"]])
display(pd.DataFrame(profile["target_associations"]))

## Search with a bounded preset

The classification preset uses scikit-learn estimators and a small candidate budget. The validation partition stays out of the search and is used only for the final score.

In [ ]:
search_artifacts = tempfile.TemporaryDirectory(prefix="sksurrogate-search-")
search = AML(
    preset="classification",
    max_evals=5,
    time_limit=30,
    cv=3,
    random_state=SEED,
    n_jobs=1,
    check_point=str(Path(search_artifacts.name)) + "/",
    verbose=0,
)
search.fit(train[features].to_numpy(), train["target"].to_numpy())
best_model = search.best_estimator_
validation_score = best_model.score(validation[features].to_numpy(), validation["target"].to_numpy())
print(f"Selected model: {best_model.__class__.__name__}")
print(f"Held-out validation score: {validation_score:.3f}")

## Evaluate and record the run

Repeated nested CV summarizes variability across repeat-level outer-fold means. With only two repeats, its normal-approximation interval is illustrative and should not be treated as precise. The validation partition remains separate from training-based model selection.

In [ ]:
workdir = tempfile.TemporaryDirectory(prefix="sksurrogate-notebook-")
tracker = mltrack(
    "notebook-scientific-workflow",
    db_name=str(Path(workdir.name) / "mltrace.db"),
)
tracker.register_data(train, "target", partition="train")
tracker.register_data(validation, "target", partition="validation")
evaluation = tracker.nested_cv_evaluation(
    best_model,
    inner_cv=2,
    outer_cv=2,
    train_partition="train",
    validation_partition="validation",
    scoring=get_scorer("accuracy"),
    repeats=2,
    confidence_level=0.95,
)
print("Repeated outer scores:", evaluation["repeated_outer_scores"])
print("Confidence interval:", evaluation["confidence_interval"])
print("Final validation score:", evaluation.get("final_validation_score"))

## Interpret the fitted model

Permutation importance is computed on the explicitly registered validation partition, with a fixed seed and bounded repeats. It describes this model on this reference sample; it is not a causal effect. The next cell exports a summary table and a two-panel figure. The interval is based on only two repeated-CV summaries, and importance error bars show permutation-repeat standard deviations, not confidence intervals. Change `REPORT_DIR` to choose another output location.

In [ ]:
import json

import matplotlib.pyplot as plt

logged_model = tracker.log_model(best_model, "bounded-preset-classifier")
importance = tracker.permutation_importance(
    logged_model,
    reference_partition="validation",
    n_repeats=3,
    max_samples=60,
    random_state=SEED,
)
importance_table = (
    importance["importances"][["feature", "importance_mean", "importance_std"]]
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
repeat_scores = np.asarray(evaluation["repeated_outer_scores"], dtype=float)
confidence_interval = evaluation["confidence_interval"]
summary_table = pd.DataFrame([
    {
        "result": "Repeated nested CV mean outer score",
        "estimate": float(repeat_scores.mean()),
        "lower": confidence_interval["lower"],
        "upper": confidence_interval["upper"],
        "scope": f"{len(repeat_scores)} repeat-level means; {confidence_interval['confidence_level']:.0%} normal-approximation CI",
    },
    {
        "result": "Held-out validation score",
        "estimate": evaluation["final_validation_score"],
        "lower": np.nan,
        "upper": np.nan,
        "scope": "separate validation partition; point estimate",
    },
]).assign(
    model=best_model.__class__.__name__,
    metric="accuracy",
    seed=SEED,
    train_rows=len(train),
    validation_rows=len(validation),
)
REPORT_DIR = Path("scientific-workflow-report")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
profile_columns = pd.DataFrame(profile["columns"]).drop(columns=["distribution"], errors="ignore")
profile_columns.to_csv(REPORT_DIR / "data-profile-columns.csv", index=False)
pd.DataFrame(profile["target_associations"]).to_csv(
    REPORT_DIR / "target-associations.csv", index=False
)
summary_table.to_csv(REPORT_DIR / "performance-summary.csv", index=False)
importance_table.to_csv(REPORT_DIR / "permutation-importance.csv", index=False)
report_metadata = {
    "dataset": "seeded sklearn make_classification synthetic data",
    "seed": SEED,
    "target": "target",
    "features": features,
    "train_rows": len(train),
    "validation_rows": len(validation),
    "profile": {
        "sampled_rows": profile["sampled_rows"],
        "feature_count": profile["feature_count"],
        "random_state": SEED,
    },
    "model": best_model.__class__.__name__,
    "metric": "accuracy",
    "nested_cv": {
        "inner_folds": 2,
        "outer_folds": 2,
        "repeats": len(repeat_scores),
        "confidence_interval": confidence_interval,
        "repeat_level_outer_scores": repeat_scores.tolist(),
    },
    "importance_reference": importance["reference"],
    "limitations": [
        "The small-sample normal-approximation interval is illustrative.",
        "Permutation-repeat standard deviations are not confidence intervals.",
        "Permutation importance is model attribution, not a causal effect.",
        "The held-out validation score is a separate point estimate.",
        "Seeds do not guarantee identical search results across runs or environments.",
    ],
}
(REPORT_DIR / "analysis-metadata.json").write_text(
    json.dumps(report_metadata, indent=2, sort_keys=True), encoding="utf-8"
)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
plot_importance = importance_table.sort_values("importance_mean", ascending=True)
axes[0].barh(
    plot_importance["feature"],
    plot_importance["importance_mean"],
    xerr=plot_importance["importance_std"],
    capsize=3,
)
axes[0].set_xlabel("Permutation score drop (mean ± repeat SD)")
axes[0].set_title("Global model attribution")
axes[0].axvline(0, color="black", linewidth=0.8)

repeat_positions = np.arange(1, len(repeat_scores) + 1)
axes[1].axvspan(
    confidence_interval["lower"],
    confidence_interval["upper"],
    alpha=0.2,
    label=f"{confidence_interval['confidence_level']:.0%} CI of repeat means",
)
axes[1].scatter(repeat_scores, repeat_positions, label="Repeat-level outer mean", zorder=3)
axes[1].axvline(repeat_scores.mean(), color="C0", label="Repeated-CV mean")
axes[1].axvline(
    evaluation["final_validation_score"],
    color="C3",
    linestyle="--",
    label="Held-out validation score",
)
axes[1].set_yticks(repeat_positions)
axes[1].set_ylabel("Nested-CV repeat")
axes[1].set_xlabel("Accuracy")
axes[1].set_title("Evaluation results (separate scopes)")
axes[1].legend(fontsize="small")
figure.suptitle(f"Seeded synthetic classification workflow (seed={SEED})")
figure.savefig(REPORT_DIR / "workflow-summary.png", dpi=300, bbox_inches="tight")
display(summary_table)
display(importance_table)
display(figure)
print("Reference:", importance["reference"])
print("Saved report files:", sorted(path.name for path in REPORT_DIR.iterdir()))
tracker.close()
workdir.cleanup()
search_artifacts.cleanup()

## Export the EDA tables and a Markdown report

The control plane renders the same blocks on the Data page. Offline, the helpers
in ``api.eda`` produce the identical tables, so the notebook report and the UI
cannot drift apart. Every table states its sample scope; the report is rendered
from the same payload the endpoint uses.


In [ ]:
# Tier 2/3 EDA tables plus a self-contained Markdown report (no API server needed).
from datetime import datetime, timezone

from api.eda import (
    effect_sizes,
    formal_tests,
    markdown_report,
    missing_mechanism,
    quality_flags,
    semantic_audit,
    stability,
)
from api.preprocessing import missing_overview, target_balance

# effect_sizes compares classes; the notebook target is a 0/1 integer column, so
# it gets an explicit string grouping instead of being read as a regression target.
eda_frame = train.assign(target_label=train["target"].astype(str))
eda_payload = {
    "task_name": "scientific-workflow",
    "partition": "train",
    "target": "target",
    "rows": len(train),
    "columns": list(train.columns),
    "ignored_columns": [],
    "feature_limit": 10,
    "row_sampling": {
        "method": "without_replacement" if profile["sampled_rows"] < len(train) else "none",
    },
    "missing": missing_overview(train, target="target", ignored=[]),
    "column_profiles": profile["columns"],
    "target_associations": profile["target_associations"],
    "target_composition": target_balance(train, target="target"),
    "quality": profile["data_quality"],
    "missingness_patterns": profile["missingness_patterns"],
    "multicollinearity": profile["multicollinearity"],
    "quality_flags": quality_flags(train, target="target", target_associations=profile["target_associations"]),
    "missing_mechanism": missing_mechanism(train, target="target"),
    "type_audit": semantic_audit(train, target="target"),
    "formal_tests": formal_tests(train, target="target"),
    "effect_sizes": effect_sizes(eda_frame, target="target", group_by="target_label", bootstrap=200, random_state=SEED),
    "stability": stability(train, target="target", random_state=SEED),
}
generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
(REPORT_DIR / "eda-report.md").write_text(
    markdown_report(eda_payload, generated_at=generated_at), encoding="utf-8"
)
pd.DataFrame(eda_payload["formal_tests"]["columns"]).to_csv(
    REPORT_DIR / "eda-formal-tests.csv", index=False
)
if eda_payload["effect_sizes"]["comparisons"]:
    pd.DataFrame(eda_payload["effect_sizes"]["comparisons"]).to_csv(
        REPORT_DIR / "eda-effect-sizes.csv", index=False
    )
else:
    print("No effect-size comparison: the target has no classes to compare.")
pd.DataFrame(eda_payload["stability"]["columns"]).to_csv(
    REPORT_DIR / "eda-stability.csv", index=False
)
if eda_payload["quality_flags"]["flags"]:
    pd.DataFrame(eda_payload["quality_flags"]["flags"]).to_csv(
        REPORT_DIR / "eda-quality-flags.csv", index=False
    )
display(pd.DataFrame(eda_payload["formal_tests"]["columns"]))
display(pd.DataFrame(eda_payload["effect_sizes"]["comparisons"]).head())
print("Wrote EDA tables and eda-report.md to", REPORT_DIR.resolve())
